# Grid network dynamics — interactive explorer

Fits **dynamical network models** to the 52-site photostim impulse-response tensor `H[stim, readout, t]` (the empirical Green's function from `cross_response.py`).

Model ladder (all keep a **node-space graph-Laplacian + diagonal sign-free B** except delay-DMD):

| model | what it is | knobs |
|---|---|---|
| **1st-order driven Laplacian** | `x_dot = -(L+diag g)x + Bu`, symmetric PSD L. Stable, interpretable, but **monotone → cannot rebound** | `ridge` |
| **2nd-order driven graph-wave** | `x_ddot = -(L+diag leak)x - Gamma x_dot + Bu`, node Laplacian on k-NN edges, sim-error fit. **Oscillates → captures the rebound** | `KNN`, `R_POD`, `maxiter` |
| **delay-DMD** | Hankel/HAVOK latent embedding. Predictive **ceiling**, not a node Laplacian | depth/rank sweep |

All the heavy lifting lives in `network_id.py`; this notebook just calls its `fit_*` functions so you can tweak knobs and see results inline. Running `python network_id.py` regenerates the full multi-panel PNGs into `network_png/`.

## Setup

In [ ]:
import sys
from pathlib import Path
import numpy as np
import scipy.linalg as sla

# make bilateral/grid importable whether the notebook runs from that dir or the repo root
HERE = Path.cwd()
GRID = HERE if (HERE / 'network_id.py').exists() else HERE / 'bilateral' / 'grid'
sys.path.insert(0, str(GRID))

import cross_response as cr
import network_id as ni          # NOTE: this forces matplotlib 'Agg' at import...
get_ipython().run_line_magic('matplotlib', 'inline')   # ...so re-enable inline AFTER importing ni
import matplotlib.pyplot as plt
print('network_id loaded from', GRID)

## Load the cached impulse-response tensor + set the fit window

`H` is `(nStim, nReadout, nWin)`. The fit window is the post-pulse autonomous decay; `PRED_EXTRA` is how far past it we test extrapolation (free-run).

In [ ]:
z = cr.load_cached()
H, sites, window = z['H'], z['sites'], z['window']
fs = float(z['fs_win']); dt = 1.0 / fs
nS = len(sites)

FIT_WIN = (0.08, 0.60); PRED_EXTRA = 0.30      # <-- tweak the fit/extrapolation window here
klo = int(np.argmin(np.abs(window - FIT_WIN[0])))
khi = int(np.argmin(np.abs(window - FIT_WIN[1])))
khi_pred = int(np.argmin(np.abs(window - (FIT_WIN[1] + PRED_EXTRA))))
tt = window[klo:khi_pred + 1]
print(f'{nS} nodes | dt={dt*1e3:.1f} ms | H {H.shape} | fit [{window[klo]:.3f},{window[khi]:.3f}]s, extrapolate to {window[khi_pred]:.3f}s')

In [ ]:
# shared evaluator: median free-run R2 (in-window), extrapolation R2, and same-site R2
def eval_freerun(frfn):
    fr, ex, sf = [], [], []
    for s in range(nS):
        traj = frfn(s, khi_pred - klo)
        act = H[s, :, klo:khi_pred + 1].T
        fr.append(ni.r2(act[:khi - klo + 1], traj[:khi - klo + 1]))
        ex.append(ni.r2(act, traj))
        sf.append(ni.r2(act[:, s], traj[:, s]))
    return np.nanmedian(fr), np.nanmedian(ex), np.nanmedian(sf)

def pooled_self(frfn, bsign):
    '''sign-normalized mean same-site response: actual vs model.'''
    sa, sm = [], []
    for s in range(nS):
        traj = frfn(s, khi_pred - klo); act = H[s, :, klo:khi_pred + 1].T
        g = np.sign(bsign[s]) or 1.0
        sa.append(g * act[:, s]); sm.append(g * traj[:, s])
    return np.array(sa), np.array(sm)

## 1 · Driven 1st-order symmetric Laplacian  — the baseline that *cannot* rebound

`x_dot = -(L + diag(gamma)) x + B u`. `L` a proper combinatorial Laplacian (undirected, non-negative weights ⇒ symmetric PSD), `gamma` a per-node leak, `B` diagonal & **sign-free** (opsin polarity is a *result*). Because `A=-(L+diag g)` is symmetric neg-def, the same-site response is a sum of decaying exponentials ⇒ **strictly monotone**.

In [ ]:
A1, L1, W1, gam1, b1 = ni.fit_driven_laplacian(H, sites, klo, khi, dt, nS, ridge=1e-2)
Md1 = sla.expm(A1 * dt)
fr1 = lambda s, n: ni.free_run(Md1, H[s, :, klo], n)
m1 = eval_freerun(fr1)
hemi = np.where(sites[:, 0] < 0, 1.0, -1.0)   # left/excit x<0 -> expect +
print(f'1st-order driven Laplacian:  free-run R2 {m1[0]:+.3f} | extrapol {m1[1]:+.3f} | self-site R2 {m1[2]:+.3f}')
print(f'diagonal-B opsin-sign recovery vs hemisphere: {np.mean(np.sign(b1)==hemi)*100:.0f}%  ({int((b1>0).sum())} +, {int((b1<0).sum())} -)')

In [ ]:
sa, sm = pooled_self(fr1, b1)
plt.figure(figsize=(6, 4))
plt.plot(tt, sa.mean(0) * 100, 'k-', lw=2, label='actual self-response (mean)')
plt.plot(tt, sm.mean(0) * 100, 'r--', lw=1.6, label='1st-order Laplacian (monotone)')
plt.axhline(0, color='.6', lw=.6); plt.axvline(window[khi], color='.5', ls=':', lw=.6)
plt.xlabel('t (s)'); plt.ylabel('sign-norm. dF/F %')
plt.title('Rebound diagnostic: first-order structurally misses the rebound'); plt.legend(); plt.show()

## 2 · Driven 2nd-order graph-wave  — node Laplacian, simulation-error, *oscillates*

`x_ddot = -(L + diag(leak)) x - Gamma x_dot + B u`. Same proper node-space Laplacian but weights live on a **k-NN local candidate edge set**; fit by **simulation error** (params in node space, trajectory simulated in an `R_POD`-mode subspace so it's cheap). Complex modes let it reproduce the biphasic dip→rebound.

**This cell runs L-BFGS — ~1–3 min.** Tweak `KNN` (neighbourhood size) and `R_POD` (subspace rank).

In [ ]:
KNN = 8; R_POD = 18; MAXITER = 120        # <-- main knobs for the wave
Ww, leakw, gw, bw, Lw, Pw, frw_fn = ni.fit_driven_wave(H, sites, klo, khi, dt, nS, r=R_POD, knn=KNN, maxiter=MAXITER)
mw = eval_freerun(frw_fn)
Aw2 = np.block([[np.zeros((nS, nS)), np.eye(nS)], [-Lw, -np.diag(gw)]])
ew = np.linalg.eigvals(Aw2)
fq = np.abs(ew.imag[np.abs(ew.imag) > 1e-3]) / (2 * np.pi)
print(f'2nd-order graph-wave:  free-run R2 {mw[0]:+.3f} | extrapol {mw[1]:+.3f} | self-site R2 {mw[2]:+.3f}')
print(f'{int((Ww>1e-6).sum())} edges, density {(Ww>1e-6).mean():.2f}; {len(fq)} oscillatory modes; top freqs {np.sort(fq)[::-1][:5].round(2)} Hz; stable {int((ew.real<=1e-6).sum())}/{2*nS}')

In [ ]:
sa2, sm2 = pooled_self(frw_fn, bw)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(tt, sa2.mean(0) * 100, 'k-', lw=2, label='actual self (mean)')
ax[0].plot(tt, sm2.mean(0) * 100, 'b--', lw=1.6, label='2nd-order graph-wave')
ax[0].axhline(0, color='.6', lw=.6); ax[0].axvline(window[khi], color='.5', ls=':', lw=.6)
ax[0].set_xlabel('t (s)'); ax[0].set_ylabel('sign-norm. dF/F %'); ax[0].legend()
ax[0].set_title(f'Graph-wave captures the rebound (self R2 {mw[2]:+.2f})')
# learned coupling on the grid, node fill = diagonal B sign
vmax = np.percentile(np.abs(bw), 98)
ax[1].scatter(sites[:, 0], sites[:, 1], c=bw, cmap='RdBu_r', s=90, ec='k', lw=.3, vmin=-vmax, vmax=vmax, zorder=3)
thr = np.percentile(Ww[Ww > 0], 90)
for i in range(nS):
    for j in range(i + 1, nS):
        if Ww[i, j] >= thr:
            ax[1].plot([sites[i,0], sites[j,0]], [sites[i,1], sites[j,1]], '0.35', lw=.5 + 2*Ww[i,j]/Ww.max(), alpha=.5, zorder=1)
ax[1].axvline(0, color='.5', ls='--', lw=.8); ax[1].set_aspect('equal'); ax[1].set_xticks([]); ax[1].set_yticks([])
ax[1].set_title('learned node Laplacian coupling (top 10% edges)')
plt.tight_layout(); plt.show()

## 3 · Delay-DMD  — predictive ceiling (latent Hankel, *not* a node Laplacian)

Sweeps delay depth × rank and picks the best by extrapolation R². This is the flexible latent model to benchmark the interpretable Laplacians against.

In [ ]:
(exm, frm, dd, drank, Ur, Atil), tbl = ni.sweep_delay(H, klo, khi, khi_pred, dt, nS)
frd = lambda s, n: ni.free_run_delay(Ur, Atil, np.concatenate([H[s, :, klo - t] for t in range(dd)]), n, nS)
md = eval_freerun(frd)
print(f'delay-DMD best d={dd}, r={drank}:  free-run R2 {md[0]:+.3f} | extrapol {md[1]:+.3f}')

## Model ladder

In [ ]:
print('MODEL LADDER   (free-run R2 / self-site R2 / node-space Laplacian?)')
print(f'  1st-order driven Laplacian   free {m1[0]:+.2f}   self {m1[2]:+.2f}   node-L YES  (monotone: cannot rebound)')
print(f'  2nd-order driven graph-wave  free {mw[0]:+.2f}   self {mw[2]:+.2f}   node-L YES  (oscillates: {len(fq)} complex modes)')
print(f'  delay-DMD (predictive ceil.) free {md[0]:+.2f}   self  n/a    node-L NO   (latent Hankel)')

## Knobs to explore

- **`FIT_WIN` / `PRED_EXTRA`** (data cell): move the fit window / extrapolation horizon.
- **`KNN`** (wave): larger ⇒ denser candidate graph, more edges to learn (8→12 is the obvious sweep).
- **`R_POD`** (wave): POD subspace rank the sim-error fit runs in (18→30 to give it more room).
- **`MAXITER`** (wave): L-BFGS iterations; raise if it hasn't converged.
- **`ridge`** (1st-order): Tikhonov toward small weights.

To narrow the wave→delay-DMD gap, try `KNN=12, R_POD=30`. To regenerate the full multi-panel figures (`net_driven_laplacian.png`, `net_driven_wave.png`, spectra, etc.) into `network_png/`, run `python network_id.py` from `bilateral/grid/`.